# BAB 1 - Data Preporcessing Sklearn
Dokumen ini berisi panduan ringkas pra-pemrosesan data (*Data Preprocessing*) menggunakan Scikit-Learn. Seluruh alur dipraktikkan menggunakan **satu DataFrame yang sama** agar transisi antar-tahap terlihat jelas.

---

## A. Menangani Data Hilang
Penanganan nilai kosong ini menggunakan `SimpleImputer`:
* **Numerik:** Menggunakan strategi `median` (cocok jika ada pencilan/*outliner*)
* **Kategorikal:** Menggunakan strategi `most_frequent` (modus/paling sering muncul)

    ```python
    imputer_num = SimpleImputer(strategy='median')
    imputer_cat = SimpleImputer(strategy='most_frequent')
    ```

### 🔍 Detail Teknis
Menambahkan `reval()` pada hasil `.fit_transform()` untuk meratakan array menjadi 1D kembali pada data yang bertipe `object` (pointer string)

```python
df['Pendidikan'] = imputer_cat.fit_transform(df[['Pendidikan']]).ravel()
```
---

## B. Encoding Data Teks ke Angka
Model Machine Learning hanya menerima input numerik, sehingga teks harus diubah:
* `OrdinalEncoder` : Digunakan untuk data kategorikal yang memiliki hirarki/tingkatan (misal: `SMA` < `S1` < `S2`)
* `OneHotEncoder` : Digunakan untuk data kategorikal nominal tanpa tingkatan (misal: `Pria`, `Wanita`)

    ```python
    mapping_pendidikan = OrdinalEncoder(categories=[["SMA", "S1", "S2"]])
    mapping_kota = OneHotEncoder(sparse_output=False)
    ```

### 🔍 Detail Teknis
Pada `OneHotEncoder` akan menghasilkan beberapa kolom (mengikuti keunikan data), sehingga harus dibungkus ke dalam `pd.DataFrame` (DataFrame) terpisah lalu digabungkan menggunakan `pd.concat(..., axis=1)`

---

## C. Pembagian Data
Data wajib dipisahkan menjadi **Data Latih** (`X_train`) dan **Data Uji** (`X_test`)

```python
X = df.drop(columns=['Target'])
y = df['Target']

# Bagi data: 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
```

---

## D. Penyetaraan Skala Fitur
Proses ini menyetarakan rentang angka pada variabel numerik agar algoritma tidak bias terhadap angka yang nominalnya jauh lebih besar
* `StandartScaler` : Mengubah rata-rata ($\mu$) menjadi $0$ dan standar deviasi ($\sigma$) menjadi $1$ (skala berkisar antara $-3$ hingga $+3$)
* `MinMaxScaler` : Mengubah skala data agar berada tepat di rentang $0$ sampai $1$

    ```python
    scaler = StandardScaler()
    kolom_num = ['Umur', 'Gaji']

    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()

    # Fit & Transform HANYA pada Data Latih
    X_train_scaled[kolom_num] = scaler.fit_transform(X_train[kolom_num])

    # HANYA Transform pada Data Uji
    X_test_scaled[kolom_num] = scaler.transform(X_test[kolom_num])
    ```

---

## E. Aturan Pencegahan Kebocoran Data (Data Leakage)
Prinsip terpenting dalam pra-pemrosesan data adalah **mencegah kebocoran informasi dari Data Uji ke Data Latih**
* **DILARANG melakukan scaling sebelum plit**
Jika scaling dilakukan pada dataset sebelum pemisahan, nilai rata-rata ($\mu$) dan standar deviasi ($\sigma$) dihitung dari gabungan data latih dan data uji. Ini menyebabkan data latih menyerap informasi statistik milik data uji (Data Leakage), layaknya siswa yang mendapat bocoran statistik soal ujian
* **Cara Kerja `.transform()` pada Data Test**
Fungsi ini meminjam nilai $\mu$ dan $\sigma$ murni hasil hitungan X_train untuk mengubah skala X_test. Hasil scaling gabungan data dan scaling terpisah tidak akan sama, karena nilai ekstrim pada data uji akan menggeser rata-rata jika digabung di awal.

In [1]:
import pandas as pd
import numpy as np

In [25]:
# Dataset
df = pd.DataFrame({
    'Pendidikan': ['SMA', 'S1', 'S2', 'S1', np.nan],
    'Kota': ['Jakarta', 'Bandung', 'Jakarta', 'Surabaya', 'Bandung'],
    'Umur': [25, 30, np.nan, 45, 35],
    'Gaji': [5000000, 12000000, 8000000, 20000000, 10000000],
    'Target_Setuju': [0, 1, 1, 1, 0]                   # Target (0 = Ditolak, 1 = Disetujui)
})

In [44]:
# Menangani Data Hilang

from sklearn.impute import SimpleImputer

# Umur (numerik) -> median
imputer_num = SimpleImputer(strategy="median")
df["Umur"] = imputer_num.fit_transform(df[["Umur"]])

# Pendidikan (kategorik) -> modus
imputer_cat = SimpleImputer(strategy='most_frequent')
df['Pendidikan'] = imputer_cat.fit_transform(df[['Pendidikan']]).ravel()
df

,Pendidikan,Kota,Umur,Gaji,Target_Setuju,Pendidikan_Encoded
0,SMA,Jakarta,25.0,5000000,0,0.0
1,S1,Bandung,30.0,12000000,1,1.0
2,S2,Jakarta,32.5,8000000,1,2.0
3,S1,Surabaya,45.0,20000000,1,1.0
4,S1,Bandung,35.0,10000000,0,1.0


In [51]:
# Mengubah Data Teks ke Angka

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# Ordinal Encoder (menghasilkan 1 kolom)
mapping_pendidikan = OrdinalEncoder(categories=[["SMA", "S1", "S2"]])
df["Pendidikan_Encoded"] = mapping_pendidikan.fit_transform(df[["Pendidikan"]])

# One-Hot Encoder (menghasilkan banyak, sesuai banyak data unik)
mapping_kota = OneHotEncoder(sparse_output=False)
kota_encoded = mapping_kota.fit_transform(df[["Kota"]]) # menghasilkan matriks

# Buat dataframe baru
df_kota_encoded = pd.DataFrame(kota_encoded, columns=mapping_kota.get_feature_names_out(["Kota"]))
# Gabungkan dataframe awal dengan dataframe baru, lalu hilangkan kolom kategorial
df_final = pd.concat([df, df_kota_encoded], axis=1).drop(columns=["Pendidikan", "Kota"])

df_final

,Umur,Gaji,Target_Setuju,Pendidikan_Encoded,Kota_Bandung,Kota_Jakarta,Kota_Surabaya
0,25.0,5000000,0,0.0,0.0,1.0,0.0
1,30.0,12000000,1,1.0,1.0,0.0,0.0
2,32.5,8000000,1,2.0,0.0,1.0,0.0
3,45.0,20000000,1,1.0,0.0,0.0,1.0
4,35.0,10000000,0,1.0,1.0,0.0,0.0


In [ ]:
# Pembagian Data (Train-Test Split)

from sklearn.model_selection import train_test_split

# Pisahkan fitur (X) dan target (y)
X = df_final.drop(columns="Target_Setuju")
y = df_final["Target_Setuju"]

# Bagi data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


((4, 6), (1, 6))

In [61]:
# Penyetaraan Skala Fitur

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Tentukan kolom fitur yang ingin di-scale
scale_fitur = ["Umur", "Gaji"]

# Duplikat dataframe agar rapi
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Fit & Transform hanya di X_train
X_train_scaled[scale_fitur] = scaler.fit_transform(X_train[scale_fitur])
# Transform hanya di X_test
X_test_scaled[scale_fitur] = scaler.transform(X_test[scale_fitur])

display(X_train_scaled, X_test_scaled)

,Umur,Gaji,Pendidikan_Encoded,Kota_Bandung,Kota_Jakarta,Kota_Surabaya
4,0.087370,-0.133235,1.0,1.0,0.0,0.0
2,-0.262111,-0.488527,2.0,0.0,1.0,0.0
0,-1.310556,-1.021466,0.0,0.0,1.0,0.0
3,1.485297,1.643228,1.0,0.0,0.0,1.0


,Umur,Gaji,Pendidikan_Encoded,Kota_Bandung,Kota_Jakarta,Kota_Surabaya
1,-0.611593,0.222058,1.0,1.0,0.0,0.0
